# Autograd (Automatic Gradient Computation)

## Definition
**Autograd** is PyTorch's automatic differentiation engine that computes gradients of tensors automatically. It enables efficient calculation of partial derivatives needed for backpropagation during neural network training without manual derivative calculations.

## Key Concepts

### Computational Graph
When operations are performed on a tensor with `requires_grad=True`, PyTorch builds a directed acyclic graph (DAG) that records:
- Every operation applied
- The order of operations
- Tensor dependencies

### Gradient (∇)
The derivative of an output with respect to an input. For example, if `loss = f(w)`, then `∂loss/∂w` is the gradient.

### Backpropagation
The process of computing gradients by traversing the computational graph backwards from output to inputs using the chain rule.

## 5 Steps of Autograd in Training

### Step 1: Prepare Tensors with `requires_grad=True`
Mark parameters that need gradient computation. Only tensors with `requires_grad=True` will accumulate gradients.

```python
w = torch.tensor([initial_value], requires_grad=True)  # Will compute ∂loss/∂w
x = torch.tensor([data_value], requires_grad=False)    # Data doesn't need gradients
```

### Step 2: Forward Pass - Build Computational Graph
Perform operations on tensors. PyTorch automatically builds the graph recording each step.

```python
y = model(x)        # Operations tracked
loss = loss_fn(y, target)  # Build complete graph
```

### Step 3: Compute Loss
Calculate a scalar loss value that represents model error.

```python
loss = torch.nn.MSELoss()(y_pred, y_true)
# loss is a scalar tensor with computational graph attached
```

### Step 4: Backward Pass - Compute Gradients
Call `backward()` on the scalar loss to compute gradients for all tensors with `requires_grad=True`.

```python
loss.backward()  # Traverse graph backward, compute all gradients
# Now w.grad contains ∂loss/∂w
```

### Step 5: Update Parameters
Use computed gradients to update parameters (usually via optimizer).

```python
with torch.no_grad():  # Don't track these updates
    w -= learning_rate * w.grad  # Gradient descent step

w.grad.zero_()  # Clear gradients for next iteration
```

## Step-by-Step Example

| Step | Action | Code | Result |
|------|--------|------|--------|
| 1️⃣ | Prepare | `x = torch.tensor([2.0], requires_grad=True)` | x tracks gradients |
| 2️⃣ | Forward | `y = x ** 2 + 3*x + 1` | Graph built: y = f(x) |
| 3️⃣ | Loss | `loss = y` | Scalar: loss = 15 (at x=2) |
| 4️⃣ | Backward | `loss.backward()` | x.grad = ∂loss/∂x = 7 |
| 5️⃣ | Update | `x -= 0.01 * x.grad` | x = 1.93 |

## Key Rules

⚠️ **Gradients Accumulate**
```python
loss1.backward()  # x.grad = 5
loss2.backward()  # x.grad += 3 → x.grad = 8 (accumulated!)
x.grad.zero_()    # Must reset explicitly
```

🔒 **Disable Gradients When Not Needed**
```python
with torch.no_grad():  # Faster, uses less memory
    y_pred = model(x)
```

🧹 **Zero Gradients Between Steps**
```python
optimizer.zero_grad()  # Or x.grad.zero_()
loss.backward()
optimizer.step()
```

## When Autograd is Active
- **During training**: ✅ Yes - compute gradients for learning
- **During inference/evaluation**: ❌ No - use `torch.no_grad()` for efficiency
- **For data tensors**: ❌ No - only for parameters

## Why Autograd Matters

| Problem | Solution |
|---------|----------|
| Manual derivatives are error-prone | Autograd computes automatically |
| Complex models have complex gradients | Autograd applies chain rule automatically |
| Numerical gradient checking is slow | Autograd is efficient |
| Enables research | Easy to try new architectures |


In [1]:
import torch

print("=" * 70)
print("STEP-BY-STEP: 5 Steps of Autograd")
print("=" * 70)

print("\n" + "="*70)
print("STEP 1: Prepare Tensors with requires_grad=True")
print("="*70)

# Create parameter that needs gradients
w = torch.tensor([0.5], requires_grad=True)
b = torch.tensor([0.0], requires_grad=True)

# Create data (doesn't need gradients)
x = torch.tensor([2.0], requires_grad=False)
y = torch.tensor([5.0], requires_grad=False)

print(f"\nParameter w: {w}, requires_grad: {w.requires_grad}")
print(f"Parameter b: {b}, requires_grad: {b.requires_grad}")
print(f"Data x: {x}, requires_grad: {x.requires_grad}")
print(f"Data y: {y}, requires_grad: {y.requires_grad}")

print("\n" + "="*70)
print("STEP 2: Forward Pass - Build Computational Graph")
print("="*70)

# Forward pass: y_pred = w*x + b
y_pred = w * x + b
print(f"\nModel: y_pred = w*x + b")
print(f"y_pred = {w.item()}*{x.item()} + {b.item()} = {y_pred.item()}")

print("\n" + "="*70)
print("STEP 3: Compute Loss")
print("="*70)

# Calculate MSE loss
loss = (y_pred - y) ** 2  # Simple squared error
print(f"\nLoss = (y_pred - y)^2")
print(f"Loss = ({y_pred.item()} - {y.item()})^2 = {loss.item()}")
print(f"Loss has grad_fn: {loss.grad_fn}")

print("\n" + "="*70)
print("STEP 4: Backward Pass - Compute Gradients")
print("="*70)

# Compute gradients
loss.backward()
print(f"\nAfter loss.backward():")
print(f"w.grad = {w.grad.item():.4f}")
print(f"b.grad = {b.grad.item():.4f}")

# Verify manually
# loss = (w*x + b - y)^2 = (0.5*2 + 0 - 5)^2 = (-4)^2 = 16
# ∂loss/∂w = 2*(w*x + b - y)*x = 2*(-4)*2 = -16
# ∂loss/∂b = 2*(w*x + b - y)*1 = 2*(-4)*1 = -8
print(f"\nManual verification (chain rule):")
print(f"∂loss/∂w = 2*(w*x + b - y)*x = 2*({y_pred.item() - y.item()})*{x.item()} = {(2*(y_pred.item() - y.item())*x.item()):.4f}")
print(f"∂loss/∂b = 2*(w*x + b - y)*1 = 2*({y_pred.item() - y.item()})*1 = {(2*(y_pred.item() - y.item())):.4f}")

print("\n" + "="*70)
print("STEP 5: Update Parameters Using Gradients")
print("="*70)

learning_rate = 0.1
print(f"\nLearning rate: {learning_rate}")
print(f"Before update: w = {w.item():.4f}, b = {b.item():.4f}")

# Update parameters (manually for clarity)
with torch.no_grad():
    w -= learning_rate * w.grad
    b -= learning_rate * b.grad

print(f"After update:  w = {w.item():.4f}, b = {b.item():.4f}")

# Zero gradients for next iteration
w.grad.zero_()
b.grad.zero_()
print(f"\nAfter zero_grad(): w.grad = {w.grad}, b.grad = {b.grad}")


STEP-BY-STEP: 5 Steps of Autograd

STEP 1: Prepare Tensors with requires_grad=True

Parameter w: tensor([0.5000], requires_grad=True), requires_grad: True
Parameter b: tensor([0.], requires_grad=True), requires_grad: True
Data x: tensor([2.]), requires_grad: False
Data y: tensor([5.]), requires_grad: False

STEP 2: Forward Pass - Build Computational Graph

Model: y_pred = w*x + b
y_pred = 0.5*2.0 + 0.0 = 1.0

STEP 3: Compute Loss

Loss = (y_pred - y)^2
Loss = (1.0 - 5.0)^2 = 16.0
Loss has grad_fn: <PowBackward0 object at 0x00000198FE3A3CD0>

STEP 4: Backward Pass - Compute Gradients

After loss.backward():
w.grad = -16.0000
b.grad = -8.0000

Manual verification (chain rule):
∂loss/∂w = 2*(w*x + b - y)*x = 2*(-4.0)*2.0 = -16.0000
∂loss/∂b = 2*(w*x + b - y)*1 = 2*(-4.0)*1 = -8.0000

STEP 5: Update Parameters Using Gradients

Learning rate: 0.1
Before update: w = 0.5000, b = 0.0000
After update:  w = 2.1000, b = 0.8000

After zero_grad(): w.grad = tensor([0.]), b.grad = tensor([0.])


In [2]:
import torch

print("\n" + "=" * 70)
print("COMPLETE TRAINING LOOP: Using Autograd Over Multiple Epochs")
print("=" * 70)

# True function: y = 3x + 2
x_data = torch.tensor([[1.0], [2.0], [3.0], [4.0]])
y_data = torch.tensor([[5.0], [8.0], [11.0], [14.0]])

print(f"\nDataset (true function: y = 3x + 2):")
print(f"x: {x_data.squeeze().tolist()}")
print(f"y: {y_data.squeeze().tolist()}")

# Initialize parameters randomly
w = torch.tensor([[0.0]], requires_grad=True)
b = torch.tensor([0.0], requires_grad=True)

learning_rate = 0.01
num_epochs = 5

print(f"\nInitial parameters: w = {w.item():.4f}, b = {b.item():.4f}")
print(f"Expected: w = 3.0, b = 2.0")
print(f"\nTraining for {num_epochs} epochs with lr = {learning_rate}:\n")

for epoch in range(num_epochs):
    print(f"{'Epoch':<6} {'w':<10} {'b':<10} {'w.grad':<12} {'b.grad':<12} {'Loss':<10}")
    print("-" * 60)
    
    # Step 2: Forward Pass
    y_pred = x_data @ w + b
    
    # Step 3: Compute Loss
    loss = ((y_pred - y_data) ** 2).mean()
    
    # Step 4: Backward Pass
    loss.backward()
    
    print(f"{epoch+1:<6} {w.item():<10.4f} {b.item():<10.4f} {w.grad.item():<12.4f} {b.grad.item():<12.4f} {loss.item():<10.4f}")
    
    # Step 5: Update Parameters
    with torch.no_grad():
        w -= learning_rate * w.grad
        b -= learning_rate * b.grad
    
    # Zero gradients
    w.grad.zero_()
    b.grad.zero_()

print(f"\nFinal parameters:    w = {w.item():.4f}, b = {b.item():.4f}")
print(f"Expected parameters: w = 3.0, b = 2.0")
print(f"✓ Autograd successfully optimized parameters!")

# Verify on test data
print(f"\nVerification on new data:")
x_test = torch.tensor([[5.0], [6.0]])
y_test = torch.tensor([[17.0], [20.0]])

with torch.no_grad():
    y_pred_test = x_test @ w + b
    mse_test = ((y_pred_test - y_test) ** 2).mean()

print(f"x_test: {x_test.squeeze().tolist()}")
print(f"y_test (actual): {y_test.squeeze().tolist()}")
print(f"y_pred (predicted): {y_pred_test.squeeze().tolist()}")
print(f"Test MSE: {mse_test.item():.6f}")



COMPLETE TRAINING LOOP: Using Autograd Over Multiple Epochs

Dataset (true function: y = 3x + 2):
x: [1.0, 2.0, 3.0, 4.0]
y: [5.0, 8.0, 11.0, 14.0]

Initial parameters: w = 0.0000, b = 0.0000
Expected: w = 3.0, b = 2.0

Training for 5 epochs with lr = 0.01:

Epoch  w          b          w.grad       b.grad       Loss      
------------------------------------------------------------
1      0.0000     0.0000     -55.0000     -19.0000     101.5000  
Epoch  w          b          w.grad       b.grad       Loss      
------------------------------------------------------------
2      0.5500     0.1900     -45.8000     -15.8700     70.4673   
Epoch  w          b          w.grad       b.grad       Loss      
------------------------------------------------------------
3      1.0080     0.3487     -38.1365     -13.2626     48.9342   
Epoch  w          b          w.grad       b.grad       Loss      
------------------------------------------------------------
4      1.3894     0.4813     -31.7

In [3]:
import torch

print("\n" + "=" * 70)
print("⚠️ IMPORTANT: Gradient Accumulation Pitfall")
print("=" * 70)

x = torch.tensor([2.0], requires_grad=True)
print(f"\nInitial: x = {x.item()}, x.grad = {x.grad}")

# First backward pass
y1 = x ** 2
y1.backward()
print(f"\nAfter first backward (y1 = x^2):")
print(f"  x.grad = {x.grad.item()} (which is 2*x = 2*2 = 4)")

# Second backward pass WITHOUT zeroing
y2 = x ** 3
y2.backward()
print(f"\nAfter second backward (y2 = x^3) WITHOUT zero_grad():")
print(f"  x.grad = {x.grad.item()} (accumulated: 4 + 12 = 16)")
print(f"  ⚠️ WRONG! Should be 3*x^2 = 3*4 = 12, not 16")

# Fix: zero gradients before next computation
print(f"\n" + "-"*70)
print(f"CORRECT way: Always zero gradients")
print(f"-"*70)

x = torch.tensor([2.0], requires_grad=True)
print(f"\nInitial: x = {x.item()}, x.grad = {x.grad}")

# First backward
y1 = x ** 2
y1.backward()
print(f"\nAfter first backward: x.grad = {x.grad.item()}")

# Zero gradients
x.grad.zero_()
print(f"After x.grad.zero_(): x.grad = {x.grad}")

# Second backward
y2 = x ** 3
y2.backward()
print(f"After second backward: x.grad = {x.grad.item()} (correct! = 12)")

print(f"\n" + "="*70)
print("KEY TAKEAWAY:")
print("="*70)
print(f"""
Always follow this pattern in training loops:

    for epoch in range(num_epochs):
        # Forward
        output = model(input)
        loss = loss_fn(output, target)
        
        # Backward
        optimizer.zero_grad()  # ← ALWAYS zero first!
        loss.backward()
        optimizer.step()
""")



⚠️ IMPORTANT: Gradient Accumulation Pitfall

Initial: x = 2.0, x.grad = None

After first backward (y1 = x^2):
  x.grad = 4.0 (which is 2*x = 2*2 = 4)

After second backward (y2 = x^3) WITHOUT zero_grad():
  x.grad = 16.0 (accumulated: 4 + 12 = 16)
  ⚠️ WRONG! Should be 3*x^2 = 3*4 = 12, not 16

----------------------------------------------------------------------
CORRECT way: Always zero gradients
----------------------------------------------------------------------

Initial: x = 2.0, x.grad = None

After first backward: x.grad = 4.0
After x.grad.zero_(): x.grad = tensor([0.])
After second backward: x.grad = 12.0 (correct! = 12)

KEY TAKEAWAY:

Always follow this pattern in training loops:

    for epoch in range(num_epochs):
        # Forward
        output = model(input)
        loss = loss_fn(output, target)

        # Backward
        optimizer.zero_grad()  # ← ALWAYS zero first!
        loss.backward()
        optimizer.step()

